# Cryptography assignment Azar/11


## Problem 1 — How many plaintext/ciphertext pairs are needed?

When performing an exhaustive key search using plaintext/ciphertext pairs, a *wrong* key will match a given plaintext/ciphertext pair with probability $1/2^n$ (because for a wrong key the cipher behaves like a random permutation). For $q$ independent known plaintext/ciphertext pairs the probability that a fixed wrong key matches **all** pairs is $2^{-n q}$.

There are about $2^k$ candidate keys, so the expected number of wrong keys that survive (match all $q$ pairs) is approximately
$$
N_{wrong \;survivors} \approx 2^k * 2^{-n q} = 2^{k - n q}.
$$
We want the expected number of surviving wrong keys to be less than $2^{-10}$ (so with high probability no wrong key remains). Thus require

$$
k - n q ≤ -10  →  n q ≥ k + 10.
$$

Plugging $k = 256$, $n = 128$:

$$
128 q ≥ 256 + 10 = 266
q ≥ 266 / 128 = 2.078125
$$

So the minimum integer $q$ is **3** plaintext/ciphertext pairs.

**Answer:** $q = 3$ pairs.


## Problem 2 — Triple DES (E–D–E) and meet-in-the-middle

**Observations:**
- This 3-stage construction uses $K1$ twice and $K2$ once. The total key material is $K1 || K2$ (effectively 112 bits if each DES key is 56 bits).  A naive exhaustive search over $(K1,K2)$ takes about $2^{112}$ DES operations.
- A meet-in-the-middle (MITM) attack tries to split the encryption into two parts and match an intermediate value so that the work is reduced at the cost of memory.

**Choice of intermediate value:**
Two obvious intermediate points are the value after the first $E_{K1}$ (call it $A$) or the value before the last $E_{K1}$ (call it $B$). Both are related:

- Let $A = E_{K1}(P)$ (the output of the first stage).
- Let $B$ be the input to the final $E_{K1}$, so $C = E_{K1}(B)$ and $B = D_{K1}(C)$.

From the full encryption $C = E_{K1}( D_{K2}( E_{K1}(P) ) )$ we can apply $D_{K1}$ to $C$ and obtain:

$$
D_{K1}(C) = D_{K2}( E_{K1}(P) ) = D_{K2}( A ).
$$

This gives an equation between a value that depends on $K1$ (the left-hand side) and one that depends on $K1$ and $K2$ (right-hand side). The standard MITM for double encryption $E_{K2}(E_{K1}(P))$ builds a table of $E_{K1}(P)$ over $K1$ and then checks $D_{K2}(C)$ over $K2$. For this $E_{K1}∘D_{K2}∘E_{K1}$ structure the reuse of $K1$ at both ends complicates a direct separation but we can still build an attack by enumerating $K1$ values and doing a second enumeration over $K2$ and $K1$ (or use multiple plaintexts to prune candidates).

**Practical MITM sketch (conceptual):**
1. For a chosen plaintext $P$ and its ciphertext $C$, compute for every possible $k1$ the value $A_k1 = E_{k1}(P)$ and store $(A_k1, k1)$ in a hash table indexed by $A_k1$.
2. For every possible $k2$ and every possible $k1'$ (or by computing some transform), compute $T_{k2,k1'} = D_{k2}( A_{k1'} )$ and check whether $E_{k1'}( T_{k2,k1'} ) == C$. This is essentially still brute-forcing many combinations, but using the table of $A_k1$ we can accelerate operations.

A more systematic way is to use *multiple* known plaintexts to filter wrongly matching key pairs. With $q$ plaintext/ciphertext pairs the probability that an incorrect key pair survives is drastically reduced.

**Complexity / trade-offs (qualitative):**
- Worst-case naive full search is $2^{112}$ operations.
- A direct MITM that treats the two occurrences of $K1$ independently does not fully separate keys, so the typical textbook MITM savings (as in double DES) does not directly reduce complexity from $2^{112}$ to $2^{57}$.
- However, *time-memory tradeoffs* and *using several known plaintexts* allow practical reduction of the search space: build a table for $A = E_{k1}(P)$ for all $k1$ (size $2^{56}$ entries), then for each candidate $k2$ compute $D_{k2}(X)$ for each stored $X$ and test whether a consistent $k1$ exists by checking the final encryption. This results in roughly $2^{56}$ memory and about $2^{56+56} = 2^{112}$ operations in the worst case if implemented naively, but with early pruning and multiple plaintexts the effective work to find the correct $(k1,k2)$ pair reduces significantly in practice.

**Conclusion:**
- The important upshot is that reusing $K1$ at both ends complicates a direct double-DES style MITM but with memory/time tradeoffs and multiple plaintexts the attack is still much faster than a naive $2^{112}$ search in practice (and there exist known cryptanalytic and practical techniques developed specifically for two-key 3DES). For a classroom assignment a conceptual description and this pseudocode plus complexity discussion are appropriate.


## Problem 3 — AES structure: AddRoundKey in initial round and MixColumns omission in final round

- **AddRoundKey in the initial round:** AES encryption is composed of a number of rounds. Each round is a combination of non-linear substitution ($SubBytes$), diffusion ($ShiftRows$, $MixColumns$) and key addition ($AddRoundKey$). The initial $AddRoundKey$ (sometimes called the initial round key whitening) mixes the plaintext with the cipher key before any non-linear substitution or diffusion occurs. This is required so that the first non-linear layer (SubBytes) acts on a key-dependent state; without the initial $AddRoundKey$ the first SubBytes would be independent of the key and it would leak simple structure. The initial key addition increases resistance to simple structural attacks and provides *key-dependent whitening*.

- **No $MixColumns$ in the final round:** The AES final round omits $MixColumns$ for two practical reasons:
  1. **Invertibility and symmetry with decryption:** The $MixColumns$ step is a linear diffusion layer that is easily inverted, but if you include $MixColumns$ in the final round you would need an extra inverse $MixColumns$ at the start of decryption, which is not how the standard AES specification is formulated. By omitting $MixColumns$ in the final round, the operations fit nicely so that decryption applies $InvShiftRows$, $InvSubBytes$, $AddRoundKey$ and then $InvMixColumns$ in the appropriate places, keeping the round count symmetric while making the final round simpler.
  2. **Practical implementation / alignment of operations:** Omitting $MixColumns$ in the last round makes the final round produce the output bytes directly after a final $AddRoundKey$. It simplifies implementation of the cipher and the key schedule (and software/hardware optimizations use this structure). Omitting $MixColumns$ does not reduce security because the earlier rounds already provided sufficient diffusion.

In short: $AddRoundKey$ on the input prevents the first non-linear layer from being key-independent; $MixColumns$ is omitted in the final round for design/implementation symmetry and simplicity while preserving invertibility and security.


## Problem 4 — Linear approximation across several rounds (sketch)

$$
U_{3,2} ⊕ U_{3,5} ⊕ U_{3,12} ⊕ X_4 ⊕ X_5 ⊕ X_6 = ⊕ K_{i,j}
$$

It asks to treat this as a linear approximation, write an attack using that linear expression, and describe the time/space/complexity and how additional rounds or round numbering affect the attack.

**High-level interpretation and attack sketch:**
- The given linear relation states that the XOR of certain intermediate bits in round 3 ($U_{3,*}$) together with some plaintext bits $X_4, X_5, X_6$ equals the XOR of some key bits. In linear cryptanalysis this is a *linear approximation* that holds with some bias ε away from 1/2 (here the problem gives ε = 2^{-6}).

- With a known linear approximation over 3 rounds that has bias $ε$, you can build an $r+1$-round distinguisher by appending one additional round at the end for which you guess the subkey bits that appear in the approximation at the last round. The general strategy:
  1. Collect $N ≈ ε^{-2}$ known plaintexts (this is the standard linear cryptanalysis data complexity approximation).
  2. For each plaintext compute the left-hand side of the linear approximation **up to the last round intermediate bits** by partially decrypting the ciphertext for the guessed subkey bits of the last round (so you guess only the small set of key bits that appear in the approximation). Partial decryption turns ciphertext bits into the $U_{3,*}$ values used in the linear relation.
  3. For each guess of the target subkey bits, compute the XOR-sum over all $N$ plaintexts of the linear expression. If the guess is correct, the observed bias (distance of sample frequency from 1/2) will be about $ε$; incorrect guesses will have near-zero bias.
  4. Select the key guess(es) with the largest bias; then verify on additional plaintexts and extend the search to recover remaining key bits as needed.

- **Complexities:**
  - **Data complexity:** $N ≈ c * ε^{-2}$ plaintexts are required to observe a bias ε with reasonable confidence (where $c$ is a small constant depending on target confidence). For ε = 2^{-6} this gives $N ≈ 2^{12} = 4096$ (up to constant factors).
  - **Time complexity:** dominated by: for each candidate subkey (the number of guessed key-bit combinations), we must partially decrypt $N$ ciphertexts and evaluate the XOR. If the approximated linear expression involves $m$ key bits then the time is about $2^m * N * cost_per_partial_decrypt$.
  - **Memory:** typically $O(N)$ to store counters or $O(1)$ if you stream and keep only counters per key guess.

**Notes about round numbering and extending to $r+1$ rounds:**
- To extend a linear approximation that covers $r$ rounds into an $r+1$ round distinguisher you typically only need to guess the small number of key bits that enter the last round of the cipher that affect the linear expression; the rest of the round-key bits are not necessary because you never fully decrypt all rounds — only partial decryption to the approximation layer is needed.

**Concrete numbers from the problem:**
- The problem gave the linear expression and said $ε = 2^{-6}$. Using the approximate rule $N ≈ ε^{-2}$ we get $N ≈ 2^{12}$ plaintexts to detect the bias. Then with $m$ guessed key bits the time cost is $≈ 2^m * 2^{12}$ partial decrypt operations.

**Conclusion:**
This answer is a sketch of how to convert the given linear equation into a real linear attack. A full worked-out attack would require specifying which S-box bits and which key bits are involved, implementing the partial decryption, and enumerating the guessed subkey bits; for an assignment setting the conceptual sketch together with the complexity estimates ($N ≈ ε^{-2}$ and time $≈ 2^m * N$) suffices.
